# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

# from blockymetamaterials.plotting import generate_animation, generate_frames, plot_geometry, generate_patch_collection, generate_mode_images
from blockymetamaterials.utils import EigenmodeData, ControlParams, GeometricalParams, LigamentParams, MechanicalParams
from blockymetamaterials.geometry import RotatedSquareGeometry
from blockymetamaterials.energy import build_strain_energy, stretching_torsional_spring_energy
from blockymetamaterials.analysis_utils import *


import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import to_rgb, ListedColormap
from tqdm import tqdm

# import nlopt
import numpy as np
import jax.numpy as jnp
from jax._src.config import config
config.update("jax_enable_x64", True)  # enable float64 type
# config.update("jax_log_compiles", 1)

# import scienceplots
# plt.style.use('science')
%matplotlib widget

plt.rc('text', usetex=True)
plt.rcParams.update({'font.size': 14})
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath} \usepackage{xcolor}'

import math, csv
from sympy import symbols, lambdify, diff


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Linear modes

## Solver

In [4]:
def set_it_up(geometry: RotatedSquareGeometry, initial_angle: float, hinge_params: MechanicalParams, clamping = None, n_modes = None, only_eigs = False):
    n1 = geometry.n1_cells
    n2 = geometry.n2_cells
    Nsq = n1*n2*4
    print('No. of squares =', Nsq)

    block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()

    # corner clamping (3 blocks for each corner)
    if clamping == 'corners':
        clamped_block_ids = np.array([0, 1, 2*n1-2, 2*n1-1, 2*n1, 4*n1-1, Nsq-4*n1, Nsq-2*n1-1, Nsq-2*n1, Nsq-2*n1+1, Nsq-2, Nsq-1])
    elif clamping == '2-corners':
        clamped_block_ids = np.array([0, 1, 2*n1-2, 2*n1-1, 2*n1, 4*n1-1])
    elif clamping == '1-corner':
        clamped_block_ids = np.array([0, 1, 2*n1])
    elif clamping == '3-corners':
        clamped_block_ids = np.array([0, 1, 2*n1-2, 2*n1-1, 2*n1, 4*n1-1, Nsq-4*n1, Nsq-2*n1, Nsq-2*n1+1])


    # Clamp 3 blocks at each corner, all 3 DOFs for each block
    if clamping is None:
        clamped_block_DOF_pairs = np.array([])
    elif clamping == 'corners':
        clamped_block_DOF_pairs = np.array([[block_id, DOF_id] for DOF_id in range(3) for block_id in clamped_block_ids])

    constrained_block_DOF_pairs = clamped_block_DOF_pairs

    print('Clamped blocks :', 0 if clamping is None else clamped_block_ids)
    print(rf'Free d.o.f. : {3*Nsq - len(constrained_block_DOF_pairs)}')

    # Initial conditions
    state0 = np.zeros((2, geometry.n_blocks, 3))

    # Setup energy function
    if geometry.bond_length!=0:
        potential_energy = build_strain_energy(bond_connectivity=bond_connectivity())
    else:
        potential_energy = build_strain_energy(bond_connectivity=bond_connectivity(),
                                       bond_energy_fn=stretching_torsional_spring_energy)
    
    control_params = ControlParams(
        geometrical_params=GeometricalParams(
            block_centroids=block_centroids(initial_angle),
            centroid_node_vectors=centroid_node_vectors(initial_angle),
            ),
        mechanical_params=hinge_params,
        )

    if only_eigs is True:
        eigs = linear_mode_analysis(
                displacement=state0[0],
                geometry=geometry,
                energy_fn=potential_energy,
                control_params=control_params,
                constrained_block_DOF_pairs=constrained_block_DOF_pairs,
                only_eigs=only_eigs
            )
        return eigs
    
    # Find lowest eigenfrequencies if n_modes is specified, otherwise find all modes
    if n_modes is not None:
        eigs = linear_softmode_analysis(
                displacement=state0[0],
                geometry=geometry,
                energy_fn=potential_energy,
                control_params=control_params,
                constrained_block_DOF_pairs=constrained_block_DOF_pairs,
                n_modes=n_modes,
            )
        return eigs

    eigs, eigm = linear_mode_analysis(
        displacement=state0[0],
        geometry=geometry,
        energy_fn=potential_energy,
        control_params=control_params,
        constrained_block_DOF_pairs=constrained_block_DOF_pairs,
    )

    eigen_solution = EigenmodeData(
        block_centroids=control_params.geometrical_params.block_centroids,
        centroid_node_vectors=control_params.geometrical_params.centroid_node_vectors,
        eigenvalues=eigs,
        fields=eigm,
    )
    return eigen_solution

## Parameters

In [5]:
# GREENSHIMS
# NOTE: Units are mm, N, s → mass in Mg

exp_config = experimental_parameters()

spacing = exp_config['spacing']
hinge_length = exp_config['hinge_length']       # Same as bond/ligament length
n1 = exp_config['n1_blocks']//2
n2 = exp_config['n2_blocks']//2

# Mechanical parameters
k_stretch = exp_config['k_stretch']
k_shear = exp_config['k_shear']
k_rot = exp_config['k_rot']

density = exp_config['density']                 # Mg/mm^2
initial_angle = 30*jnp.pi/180                   # dilational case
# initial_angle = 5*jnp.pi/180                  # conventional case

filename = f'{2*n1}x{2*n2}_initialangle_{initial_angle*180/np.pi:.1f}_clamped'


# ==============================================================================
# LINEAR MODEL WITH LARGER SIZE AND LOWER k_rot (MORE CONFORMAL)
# ==============================================================================
# n1 = 3*n1 
# n2 = 3*n2
# k_rot = k_rot/100
# filename = f'krotby100_{2*n1}x{2*n2}_initialangle_{initial_angle*180/np.pi:.1f}_clamped'


# Elastic modulii
B, G1, G2, M = RSmoduli(initial_angle, spacing, k_rot, k_stretch, k_shear, hinge_length)
print(B,G2,G1,M)

# Geometry
geometry = RotatedSquareGeometry(n1_cells=n1, n2_cells=n2, spacing=spacing, bond_length=hinge_length)
# centroid_node_vectors is a function of the initial angle
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()

greenshim_params = MechanicalParams(
        bond_params=LigamentParams(
            reference_vector=reference_bond_vectors(),
            k_stretch=k_stretch,
            k_shear=k_shear,
            k_rot=k_rot,
        ),
        density=density,
    )

n1_blocks=geometry.n1_blocks
n2_blocks=geometry.n2_blocks

0.11049093957559378 0.414705 10.29412 64.9950547764606


## Find Eigenmodes

In [6]:
clamping = 'corners'

eigdata = set_it_up(geometry, initial_angle, greenshim_params, clamping)

frequencies = np.sqrt(eigdata.eigenvalues)/(2*np.pi)
print('Mode frequencies (Hz) - ', frequencies[:10])

No. of squares = 384
Clamped blocks : [  0   1  22  23  24  47 336 359 360 361 382 383]
Free d.o.f. : 1116
Mode frequencies (Hz) -  [13.80760912 15.08862008 24.5984327  27.62828607 36.01738275 36.61633202
 38.88831669 43.84259061 47.30071719 51.56763065]


### Save Data

In [ ]:
np.savez(f'../../data/linear_data/linear_eigenmode_analysis_undamped/{filename}.npz', 
         block_centroids=eigdata.block_centroids, 
         centroid_node_vectors=eigdata.centroid_node_vectors, 
         eigenvalues=eigdata.eigenvalues, 
         fields=eigdata.fields)

### Mode Profiles

In [ ]:
filename = r'linear_discrete_model_modeprofiles'
var = r'f'
scale = 100

image_data = EigenmodeData(
        block_centroids=eigdata.block_centroids,
        centroid_node_vectors=eigdata.centroid_node_vectors,
        eigenvalues=frequencies,
        fields=-eigdata.fields,
    )

xlim, ylim = geometry.get_xy_limits(initial_angle) + 1.*geometry.spacing * jnp.array([-1, 1])

# cmap rebalancing function
def gamma_adjusted_cmap(base_cmap_name='plasma', gamma=0.5, n_colors=256):
    base = mpl.colormaps[base_cmap_name].resampled(n_colors)
    # Remap input range through gamma
    indices = np.linspace(0, 1, n_colors) ** gamma
    new_colors = base(indices)
    return ListedColormap(new_colors, name=f'{base_cmap_name}_gamma{gamma}')

# Create adjusted cmap
my_cmap = gamma_adjusted_cmap(gamma=0.7)

generate_mode_images(
    data=image_data,
    field="u",
    scale_deformation=scale,
    deformed=True,
    mode_range=jnp.arange(5),
    out_dir=f"../../data/linear_data/analytical_linear_conformal_modes/{geometry.n1_blocks}x{geometry.n2_blocks}_clamped12blocks_test/{filename}",
    figsize=(6, 4),
    xlim=xlim,
    ylim=ylim,
    geometry=geometry,
    mesh=False,
    lattice=True,
    latt_alpha=1,
    cmap=my_cmap,
    var=var
)

/var/folders/zt/h8fvtmh16r7_v94bq65qv8340000gn/T/ipykernel_2147/4127861075.py:17: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  base = cm.get_cmap(base_cmap_name, n_colors)
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed x limits to fulfill fixed data aspect with adjustable data l

### Strains

In [ ]:
mode_range = np.arange(0,100)
cell_disps, cell_masks = unitcell_data(eigdata, geometry, mode_range)
grad, d, s1, s2, stot, r = unitcell_strain(cell_disps, cell_masks, spacing)

say('strains calculated')

# Save as npzs
np.savez(f'../../data/linear_data/strains_linear_eigenmodes/(grad,diln,s1,s2,stot,rotn)_{filename}_100modes.npz', 
         grad=grad, d=d, s1=s1, s2=s2, stot=stot, r=r)

(1116, 16, 24, 2) (100, 16, 24, 2) (16, 24)


# Find Periodic Response

In [12]:
exp_config = experimental_parameters()

lin_freqs = 2+0.2*np.arange(300)
w = lin_freqs*(2*np.pi)

angle30 = 30*np.pi/180
angle5 = 5*np.pi/180
initial_angle = angle5

n1_blocks = exp_config['n1_blocks']
n2_blocks = exp_config['n2_blocks']
Nsq = n1_blocks*n2_blocks

geometry = RotatedSquareGeometry(n1_cells=n1_blocks//2, n2_cells=n2_blocks//2, spacing=exp_config['spacing'], bond_length=exp_config['hinge_length'])
block_centroids, centroid_node_vectors, bond_connectivity, reference_bond_vectors = geometry.get_parametrization()

greenshim_params = MechanicalParams(
        bond_params=LigamentParams(
            reference_vector=reference_bond_vectors(),
            k_stretch=exp_config['k_stretch'],
            k_shear=exp_config['k_shear'],
            k_rot=exp_config['k_rot'],
        ),
        density=exp_config['density'],
    )

# Hinge Damping Model
damptype = exp_config['damptype']
dampcoeffs = np.array([exp_config['n_stretch'], exp_config['n_shear'], exp_config['n_rot']])

clampedblocks = np.array([0, 1, n1_blocks-2, n1_blocks-1, n1_blocks, 2*n1_blocks-1, Nsq-2*n1_blocks, Nsq-n1_blocks-1, Nsq-n1_blocks, Nsq-n1_blocks+1, Nsq-2, Nsq-1])

# Displacement-based driving
drive_amplitude = 0.01  # mm
drive_angle = - 45*jnp.pi/180
drive_type = 'corner'
corner_driven_blocks = np.array([(n2_blocks-2)*n1_blocks, (n2_blocks-1)*n1_blocks, (n2_blocks-1)*n1_blocks+1])
driven_block_DOF_pairs = np.array([np.array([block_id, DOF_id]) for DOF_id in range(2) for block_id in corner_driven_blocks])

constrained_block_DOF_pairs = np.concatenate((np.array([np.array([block_id, 2]) for block_id in corner_driven_blocks]),
                                              np.array([np.array([block_id, DOF_id]) for DOF_id in range(3) for block_id in clampedblocks if block_id not in corner_driven_blocks])))


print(f'corner loading | d.o.f. constrained : {len(constrained_block_DOF_pairs)}')

# Initial conditions
state0 = np.zeros((2, geometry.n_blocks, 3))

# Setup energy function
potential_energy = build_strain_energy(bond_connectivity=bond_connectivity())

control_params = ControlParams(
    geometrical_params=GeometricalParams(
        block_centroids=block_centroids(initial_angle),
        centroid_node_vectors=centroid_node_vectors(initial_angle),
    ),
    mechanical_params= greenshim_params,
)

sol = linear_response_analysis(
    frequencies=w,
    displacement=state0[0],
    geometry=geometry,
    energy_fn=potential_energy,
    control_params=control_params,
    constrained_block_DOF_pairs=constrained_block_DOF_pairs,
    drive_amplitude=drive_amplitude,
    driven_block_DOF_pairs=driven_block_DOF_pairs,
    drive_angle=drive_angle,
    damp_coeffs=dampcoeffs,
    damptype=damptype
)

data = EigenmodeData(block_centroids=control_params.geometrical_params.block_centroids,
    centroid_node_vectors=control_params.geometrical_params.centroid_node_vectors,
    eigenvalues=w,
    fields=sol
)


corner loading | d.o.f. constrained : 30
computing constrained energy
computing stiffness matrix
computing damping matrix
Driven DOF ids :  [1008 1080 1083 1009 1081 1084]
6
Drive :  [ 0.00707107  0.00707107  0.00707107 -0.00707107 -0.00707107 -0.00707107]
No. of clamped dofs: 30
No. of driven dofs: 6
b: [ 0.          0.          0.         ... -0.00707107 -0.00707107
 -0.00707107]   b shape: (1128,)
solution set shape (300, 1122)


In [ ]:
filename = f'{n1_blocks}x{n2_blocks}_initialangle_{initial_angle*180/np.pi:.1f}_clamped_freqrange(2,60,0.2)Hz'

np.savez(f'../../data/linear_data/linear_response_periodic_steadystate/{filename}.npz', 
         block_centroids=data.block_centroids, 
         centroid_node_vectors=data.centroid_node_vectors, 
         eigenvalues=data.eigenvalues, 
         fields=data.fields)